In [ ]:
#fix

import numpy as np
import tensorflow as tf
from tensorflow.keras import layers, models
from tensorflow.keras.applications import MobileNetV2
import matplotlib.pyplot as plt

print("TensorFlow Version:", tf.__version__)

# Simulasi Dataset Kustom: 5 Kelas, masing-masing 100 gambar berukuran 96x96 piksel (RGB = 3 channel)
num_samples = 500
channels = 3
img_rows, img_cols = 96, 96
num_classes = 5

# Membuat data gambar random (angka antara 0-255)
np.random.seed(42)
X_data = np.random.randint(0, 256, (num_samples, img_rows, img_cols, channels)).astype('float32')
# Normalisasi piksel menjadi rentang 0 sampai 1 agar CNN stabil
X_data /= 255.0

# Membuat label acak seimbang untuk 5 kelas (0, 1, 2, 3, 4)
y_data = np.random.randint(0, num_classes, num_samples)

# Membagi data menjadi 80% Train dan 20% Test
from sklearn.model_selection import train_test_split
X_train, X_test, y_train, y_test = train_test_split(X_data, y_data, test_size=0.2, random_state=42)

print(f"Dataset berhasil disiapkan!")
print(f"Data Latih (Train) : {X_train.shape} | Label: {y_train.shape}")
print(f"Data Uji (Test)   : {X_test.shape}  | Label: {y_test.shape}")

In [ ]:
print("Membangun model CNN biasa dari nol...")

model_cnn_biasa = models.Sequential([
    layers.Conv2D(32, (3, 3), activation='relu', input_shape=(96, 96, 3)),
    layers.MaxPooling2D((2, 2)),
    layers.Conv2D(64, (3, 3), activation='relu'),
    layers.MaxPooling2D((2, 2)),
    layers.Flatten(),
    layers.Dense(64, activation='relu'),
    layers.Dense(5, activation='softmax') # Output 5 kelas
])

model_cnn_biasa.compile(optimizer='adam',
                        loss='sparse_categorical_crossentropy',
                        metrics=['accuracy'])

# Train model cukup 5 epoch saja untuk simulasi tugas agar cepat selesai
print("Melatih CNN Biasa...")
history_biasa = model_cnn_biasa.fit(X_train, y_train, epochs=5, validation_data=(X_test, y_test), batch_size=32)

In [ ]:
print("Membangun model Transfer Learning dengan MobileNetV2...")

# 1. Ambil base model MobileNetV2 tanpa lapisan klasifikasi atasnya (include_top=False)
base_model = MobileNetV2(input_shape=(96, 96, 3), include_top=False, weights='imagenet')

# 2. Bekukan (Freeze) semua lapisan dasar agar kepintaran aslinya tidak rusak
base_model.trainable = False

# 3. Satukan base model dengan lapisan klasifikasi kustom baru kita
model_transfer = models.Sequential([
    base_model,
    layers.GlobalAveragePooling2D(),
    layers.Dense(64, activation='relu'),
    layers.Dropout(0.2), # Mencegah overfitting
    layers.Dense(5, activation='softmax') # Menyesuaikan target 5 kelas kita
])

model_transfer.compile(optimizer='adam',
                    loss='sparse_categorical_crossentropy',
                    metrics=['accuracy'])

print("Melatih Model Transfer Learning...")
history_transfer = model_transfer.fit(X_train, y_train, epochs=5, validation_data=(X_test, y_test), batch_size=32)

In [ ]:
# Mengambil histori akurasi validasi dari kedua model
acc_biasa = history_biasa.history['val_accuracy']
acc_transfer = history_transfer.history['val_accuracy']
epochs_range = range(1, 6)

# Membuat Plot Grafik Line Chart
plt.figure(figsize=(10, 5))
plt.plot(epochs_range, acc_biasa, marker='o', linestyle='-', color='red', label='CNN Biasa (Dari Nol)')
plt.plot(epochs_range, acc_transfer, marker='s', linestyle='--', color='green', label='Transfer Learning (MobileNetV2)')

plt.title('Perbandingan Akurasi Validasi: CNN Biasa vs Transfer Learning')
plt.xlabel('Epoch (Iterasi Latihan)')
plt.ylabel('Akurasi Validasi')
plt.xticks(epochs_range)
plt.grid(True, linestyle=':', alpha=0.6)
plt.legend()
plt.ylim(0, 1.1)
plt.show()